# KCV-TTS Finetuning di Kaggle (2x T4, DDP, W&B)
Notebook ini menyiapkan pipeline penuh: venv, install dependency (termasuk `causal-conv1d` dan `mamba-ssm`), clone repo, prepare dataset CSV -> Arrow, lalu finetuning dengan 2 GPU T4 menggunakan `accelerate` + W&B.

## 1) Isi Konfigurasi
Ubah nilai di cell berikut sesuai repo dan path dataset kamu di Kaggle.

In [1]:
import os
from pathlib import Path

# ===== WAJIB DIISI =====
REPO_URL = "https://github.com/AneKazek/malesbgt.git"
REPO_BRANCH = "main"
WANDB_API_KEY = "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"
WANDB_ENTITY = "haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember"
WANDB_PROJECT = "kcvanguard"

# CSV metadata sumber di Kaggle (header: audio_file|text, path audio absolut)
# Contoh: /kaggle/input/your-dataset/metadata.csv
SRC_METADATA_CSV = "/kaggle/input/datasets/anekazek/indonesian-voice-dataset/datasetku/metadata.csv"

# Nama dataset hasil prep (akan jadi data/{DATASET_NAME}_pinyin)
DATASET_NAME = "datasetku"

# Output training
MODEL_NAME = "F5TTS_MAMBA_kaggle_full"
TARGET_SAMPLE_RATE = 24000

# Training optimization untuk 2x T4
BATCH_SIZE_PER_GPU = 512
MAX_SAMPLES = 2
GRAD_ACCUM = 4
LEARNING_RATE = 1.0e-5
NUM_WARMUP_UPDATES = 200
EPOCHS = 30
NUM_WORKERS = 4

# Path kerja
WORKDIR = Path('/kaggle/working')
VENV_DIR = WORKDIR / '.venv'
REPO_DIR = WORKDIR / 'kcv-tts'

assert WANDB_API_KEY != "PASTE_WANDB_API_KEY_DI_SINI", "Isi WANDB_API_KEY dulu"
assert REPO_URL != "https://github.com/<username>/<repo>.git", "Isi REPO_URL dulu"
assert Path(SRC_METADATA_CSV).exists(), f"metadata tidak ditemukan: {SRC_METADATA_CSV}"

# Export ke environment agar tersedia di semua cell %%bash
os.environ['REPO_URL'] = REPO_URL
os.environ['REPO_BRANCH'] = REPO_BRANCH
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['WANDB_ENTITY'] = WANDB_ENTITY
os.environ['WANDB_PROJECT'] = WANDB_PROJECT
os.environ['WANDB_MODE'] = 'online'
os.environ['SRC_METADATA_CSV'] = SRC_METADATA_CSV
os.environ['DATASET_NAME'] = DATASET_NAME
os.environ['MODEL_NAME'] = MODEL_NAME
os.environ['TARGET_SAMPLE_RATE'] = str(TARGET_SAMPLE_RATE)
os.environ['BATCH_SIZE_PER_GPU'] = str(BATCH_SIZE_PER_GPU)
os.environ['MAX_SAMPLES'] = str(MAX_SAMPLES)
os.environ['GRAD_ACCUM'] = str(GRAD_ACCUM)
os.environ['LEARNING_RATE'] = str(LEARNING_RATE)
os.environ['NUM_WARMUP_UPDATES'] = str(NUM_WARMUP_UPDATES)
os.environ['EPOCHS'] = str(EPOCHS)
os.environ['NUM_WORKERS'] = str(NUM_WORKERS)

print('Config OK')
print('SRC_METADATA_CSV =', SRC_METADATA_CSV)

Config OK
SRC_METADATA_CSV = /kaggle/input/datasets/anekazek/indonesian-voice-dataset/datasetku/metadata.csv


In [2]:
!nvidia-smi

Wed Mar 25 16:21:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2) Build Venv + Clone Repo + Install Dependencies

In [3]:
%%bash
apt-get update -qq
apt-get install -y python3.11 python3.11-venv python3.11-distutils > /dev/null
python3.11 --version

Python 3.11.15


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
%%bash
set -euo pipefail
cd /kaggle/working

# Isolate from host/site customizations that can break venv bootstrap.
export PYTHONNOUSERSITE=1
unset PYTHONPATH || true
unset PYTHONHOME || true

# Require Python 3.11 to match local ABI/wheels exactly.
if command -v python3.11 >/dev/null 2>&1; then
  PY311=python3.11
else
  echo "python3.11 tidak tersedia di runtime ini. Ganti Kaggle image/runtime yang punya Python 3.11."
  exit 1
fi

# Build venv (fallback to virtualenv if stdlib venv fails).
if ! "$PY311" -m venv .venv; then
  "$PY311" -m pip install --upgrade pip
  "$PY311" -m pip install virtualenv
  "$PY311" -m virtualenv .venv
fi

source .venv/bin/activate
python - <<'PY'
import sys
assert sys.version_info[:2] == (3, 11), f"Butuh Python 3.11, dapat: {sys.version}"
print('python =', sys.version.split()[0])
PY

python -m pip install --upgrade pip setuptools wheel wrapt

# Clone repo
: "${REPO_BRANCH:=main}"
: "${REPO_URL:?REPO_URL is not set. Run Cell 3 (config) first.}"
if [ -d kcv-tts ]; then rm -rf kcv-tts; fi
git clone --depth 1 --branch "${REPO_BRANCH}" "${REPO_URL}" kcv-tts

cd kcv-tts

# Match stack exactly: torch 2.6/cu124 + project deps
test -f requirements-py311-torch260-cu124.txt
pip install -r requirements-py311-torch260-cu124.txt
pip install -e . --no-deps


python = 3.11.15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 5.2 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 79.0.1
    Uninstalling setuptools-79.0.1:
      Successfully uninstalled setuptools-79.0.1
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu124
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing met

Cloning into 'kcv-tts'...


In [5]:
# Requested order: causal first, then mamba
# Requested order: causal first, then mamba
!/kaggle/working/.venv/bin/pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+cu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.3/287.3 MB 91.0 MB/s  0:00:03:00:0100:01


In [6]:
!/kaggle/working/.venv/bin/pip install https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.3/533.3 MB 62.6 MB/s  0:00:05:00:0100:01


In [7]:
%%bash
set -e

VENV=/kaggle/working/.venv

"$VENV/bin/python" -V
"$VENV/bin/python" -c "import sys; print(sys.executable)"
"$VENV/bin/pip" -V

"$VENV/bin/pip" install --force-reinstall numpy==1.26.4 scipy==1.13.1

"$VENV/bin/python" - <<'PY'
from importlib import metadata
import sys

print("PYTHON:", sys.executable)
print("VERSION:", sys.version)

pkgs = [
    'torch','torchaudio','torchvision',
    'numpy','scipy','accelerate','hydra-core',
    'causal-conv1d','mamba-ssm','bitsandbytes'
]
for p in pkgs:
    try:
        print(p, metadata.version(p))
    except metadata.PackageNotFoundError:
        print(p, 'MISSING')
PY

Python 3.11.15
/kaggle/working/.venv/bin/python
pip 26.0.1 from /kaggle/working/.venv/lib/python3.11/site-packages/pip (python 3.11)
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 109.5 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: scipy
    Found existing installation: scipy 1.17.1
    Uninstalling scipy-1.17.1:
      Successfully uninstalled scipy-1.17.1

PYTHON: /kaggle/working/.venv/bin/python
VERSION: 3.11.15 (main, Mar  3 2026, 09:26:23) [GCC 11.4.0]
torch 2.6.0+cu124
torchaudio 2.6.0+cu124
torchvision 0.21.0+cu124
numpy 1.26.4
scipy 1.13.1
accelerate 1.13.0
hydra-core 1.3.2
causal-conv1d 1.6.1
mamba-ssm 2.3.1
bitsandbytes 0.49.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
f5-tts 1.1.18 requires gradio>=6.0.0, which is not installed.


In [8]:
%%bash
set -euo pipefail
source /kaggle/working/.venv/bin/activate
python - <<'PY'
import os
import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])
print('W&B login OK')
PY

W&B login OK


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: haidarmuhammaddzaky (haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 3) Prepare Dataset (CSV -> Arrow)

In [9]:
import csv
from pathlib import Path

src = Path(SRC_METADATA_CSV)
dst = Path('/kaggle/working/metadata_abs.csv')

with src.open('r', encoding='utf-8-sig', newline='') as fin, dst.open('w', encoding='utf-8', newline='') as fout:
    r = csv.reader(fin, delimiter='|')
    w = csv.writer(fout, delimiter='|', lineterminator='\n')
    header = next(r, None)
    if not header or header[0].strip() != 'audio_file' or header[1].strip() != 'text':
        raise ValueError('Header CSV harus: audio_file|text')
    w.writerow(['audio_file', 'text'])
    n = 0
    for row in r:
        if len(row) < 2:
            continue
        audio = row[0].strip()
        text = row[1].strip()
        if not audio or not text:
            continue
        p = Path(audio).expanduser()
        if not p.is_absolute():
            # Jika relatif, jadikan relatif terhadap folder CSV sumber
            p = (src.parent / p).resolve()
        if p.exists():
            w.writerow([p.as_posix(), text])
            n += 1

print('metadata_abs.csv rows =', n)
print('saved at', dst)

metadata_abs.csv rows = 4972
saved at /kaggle/working/metadata_abs.csv


In [10]:
%%bash
set -e

TARGET_DIR="/kaggle/working/kcv-tts/data/Emilia_ZH_EN_pinyin"
TMP_DIR="/kaggle/working/hf_tmp_emilia"

pip install -q huggingface_hub

python - <<'PY'
from huggingface_hub import snapshot_download
import os, shutil

target = "/kaggle/working/kcv-tts/data/Emilia_ZH_EN_pinyin"
tmpdir = "/kaggle/working/hf_tmp_emilia"

snapshot_download(
    repo_id="mrfakename/E2-F5-TTS",
    repo_type="space",
    revision="27cee60c0890d22dab124730a73d5453fc8359a5",
    allow_patterns=["data/Emilia_ZH_EN_pinyin/*"],
    local_dir=tmpdir,
    local_dir_use_symlinks=False,
)

src = os.path.join(tmpdir, "data", "Emilia_ZH_EN_pinyin")
os.makedirs(target, exist_ok=True)

for name in os.listdir(src):
    s = os.path.join(src, name)
    d = os.path.join(target, name)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)

print("done:", target)
PY

ls -lah "$TARGET_DIR"
test -f "$TARGET_DIR/vocab.txt"

done: /kaggle/working/kcv-tts/data/Emilia_ZH_EN_pinyin
total 24K
drwxr-xr-x 2 root root 4.0K Mar 25 16:26 .
drwxr-xr-x 3 root root 4.0K Mar 25 16:26 ..
-rw-r--r-- 1 root root  14K Mar 25 16:26 vocab.txt


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00,  6.54it/s]


In [11]:
%%bash
set -euxo pipefail
source /kaggle/working/.venv/bin/activate
cd /kaggle/working/kcv-tts

VENV_PY="/kaggle/working/.venv/bin/python"
DATASET_NAME="${DATASET_NAME:-datasetku}"
META_CSV="/kaggle/working/metadata_abs.csv"
OUT_DIR="/kaggle/working/kcv-tts/data/${DATASET_NAME}_pinyin"

test -x "$VENV_PY"
test -f "$META_CSV"

echo "DATASET_NAME=$DATASET_NAME"
echo "META_CSV=$META_CSV"
echo "OUT_DIR=$OUT_DIR"
head -n 3 "$META_CSV"

"$VENV_PY" src/f5_tts/train/datasets/prepare_csv_wavs.py "$META_CSV" "$OUT_DIR" --workers 8
ls -lah "$OUT_DIR"

DATASET_NAME=datasetku
META_CSV=/kaggle/working/metadata_abs.csv
OUT_DIR=/kaggle/working/kcv-tts/data/datasetku_pinyin
audio_file|text
/kaggle/input/datasets/anekazek/indonesian-voice-dataset/datasetku/wavs/KCVTTS-0000.wav|Reaksi berbeda disampaikan penasihat hukum pengungsi TimTim pro-integrasi Suhardi Somomoeljono.
/kaggle/input/datasets/anekazek/indonesian-voice-dataset/datasetku/wavs/KCVTTS-0001.wav|Konflik eksekutif-legislatif sebenarnya adalah hal yang lumrah dalam kehidupan berdemokrasi.

Processing 4972 audio files using 8 workers...

Saving to /kaggle/working/kcv-tts/data/datasetku_pinyin ...

For datasetku_pinyin, sample count: 4972
For datasetku_pinyin, vocab size is: 68
For datasetku_pinyin, total 6.44 hours
total 2.0M
drwxr-xr-x 2 root root 4.0K Mar 25 16:26 .
drwxr-xr-x 4 root root 4.0K Mar 25 16:26 ..
-rw-r--r-- 1 root root  38K Mar 25 16:26 duration.json
-rw-r--r-- 1 root root 1.9M Mar 25 16:26 raw.arrow
-rw-r--r-- 1 root root  14K Mar 25 16:26 vocab.txt


+ source /kaggle/working/.venv/bin/activate
++ deactivate nondestructive
++ '[' -n '' ']'
++ '[' -n '' ']'
++ hash -r
++ '[' -n '' ']'
++ unset VIRTUAL_ENV
++ unset VIRTUAL_ENV_PROMPT
++ '[' '!' nondestructive = nondestructive ']'
++ VIRTUAL_ENV=/kaggle/working/.venv
++ export VIRTUAL_ENV
++ _OLD_VIRTUAL_PATH=/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/kaggle/working/.venv/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export PATH
++ '[' -n '' ']'
++ '[' -z '' ']'
++ _OLD_VIRTUAL_PS1=
++ PS1='(.venv) '
++ export PS1
++ VIRTUAL_ENV_PROMPT='(.venv) '
++ export VIRTUAL_ENV_PROMPT
++ hash -r
+ cd /kaggle/working/kcv-tts
+ VENV_PY=/kaggle/working/.venv/bin/python
+ DATASET_NAME=datasetku
+ META_CSV=/kaggle/working/metadata_abs.csv
+ OUT_DIR=/kaggle/working/kcv-tts/data/datasetku_pinyin
+ test -x /kagg

## 4) Download Pretrained Checkpoint ke Save Dir (untuk Finetune)

In [12]:
%%bash
set -euxo pipefail
source /kaggle/working/.venv/bin/activate
cd /kaggle/working/kcv-tts

DATASET_NAME="${DATASET_NAME:-datasetku}"
MODEL_NAME="${MODEL_NAME:-F5TTS_MAMBA_kaggle_full}"
CKPT_DIR="ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
mkdir -p "$CKPT_DIR"

python - <<'PY'
from cached_path import cached_path
import shutil
from pathlib import Path
import os

model_name = os.environ.get('MODEL_NAME', 'F5TTS_MAMBA_kaggle_full')
dataset_name = os.environ.get('DATASET_NAME', 'datasetku')
ckpt_dir = Path(f'/kaggle/working/kcv-tts/ckpts/{model_name}_vocos_pinyin_{dataset_name}')
ckpt_dir.mkdir(parents=True, exist_ok=True)

src = Path(str(cached_path('hf://SWivid/F5-TTS/F5TTS_v1_Base/model_1250000.safetensors')))
dst = ckpt_dir / 'pretrained_model_1250000.safetensors'
if not dst.exists():
    shutil.copy2(src, dst)
print('pretrained ckpt ready at', dst)
PY

ls -lah "$CKPT_DIR"

pretrained ckpt ready at /kaggle/working/kcv-tts/ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku/pretrained_model_1250000.safetensors
total 1.3G
drwxr-xr-x 2 root root 4.0K Mar 25 16:26 .
drwxr-xr-x 3 root root 4.0K Mar 25 16:26 ..
-rw-r--r-- 1 root root 1.3G Mar 25 16:26 pretrained_model_1250000.safetensors


+ source /kaggle/working/.venv/bin/activate
++ deactivate nondestructive
++ '[' -n '' ']'
++ '[' -n '' ']'
++ hash -r
++ '[' -n '' ']'
++ unset VIRTUAL_ENV
++ unset VIRTUAL_ENV_PROMPT
++ '[' '!' nondestructive = nondestructive ']'
++ VIRTUAL_ENV=/kaggle/working/.venv
++ export VIRTUAL_ENV
++ _OLD_VIRTUAL_PATH=/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/kaggle/working/.venv/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export PATH
++ '[' -n '' ']'
++ '[' -z '' ']'
++ _OLD_VIRTUAL_PS1=
++ PS1='(.venv) '
++ export PS1
++ VIRTUAL_ENV_PROMPT='(.venv) '
++ export VIRTUAL_ENV_PROMPT
++ hash -r
+ cd /kaggle/working/kcv-tts
+ DATASET_NAME=datasetku
+ MODEL_NAME=F5TTS_MAMBA_kaggle_full
+ CKPT_DIR=ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_datasetku
+ mkdir -p ckpts/F5TTS_MAMBA_kaggle_full_vocos_pinyin_dat

## 5) Finetune Full di 2x T4 (DDP) + W&B

In [13]:
# %%bash
# set -euo pipefail
# source /kaggle/working/.venv/bin/activate

# python - <<'PY'
# import torch
# print("torch:", torch.__version__, "cuda:", torch.version.cuda)
# print("cxx11abi:", torch._C._GLIBCXX_USE_CXX11_ABI)
# PY

# pip uninstall -y causal-conv1d mamba-ssm || true
# pip cache purge || true

# # 1) Coba wheel yang match torch2.6 + cu12 + cp311 + cxx11abiTRUE
# pip install -U --no-deps \
#   https://github.com/state-spaces/mamba/releases/download/v2.3.1/causal_conv1d-1.5.0.post8+cu12torch2.6cxx11abiTRUE-cp311-cp311-linux_x86_64.whl \
#   https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.6cxx11abiTRUE-cp311-cp311-linux_x86_64.whl \
# || {
#   echo "[warn] direct wheel install gagal, fallback ke PyPI build"
#   pip install -U --no-build-isolation causal-conv1d==1.5.0.post8 mamba-ssm==2.3.1
# }

# python - <<'PY'
# import torch, causal_conv1d, mamba_ssm
# print("OK torch", torch.__version__)
# print("OK causal_conv1d", getattr(causal_conv1d, "__version__", "n/a"))
# print("OK mamba_ssm", getattr(mamba_ssm, "__version__", "n/a"))
# PY

In [14]:
# %%bash
# set -euo pipefail
# source /kaggle/working/.venv/bin/activate

# echo "[A] cek torch ABI"
# python - <<'PY'
# import torch
# print("torch:", torch.__version__, "cuda:", torch.version.cuda, "cxx11abi:", torch._C._GLIBCXX_USE_CXX11_ABI)
# PY

# echo "[B] uninstall + bersihkan sisa file"
# pip uninstall -y causal-conv1d mamba-ssm || true
# python - <<'PY'
# import site, glob, os, shutil
# for d in site.getsitepackages():
#     for p in glob.glob(os.path.join(d, "causal_conv1d*")) + glob.glob(os.path.join(d, "mamba_ssm*")):
#         if os.path.isdir(p):
#             shutil.rmtree(p, ignore_errors=True)
#         else:
#             try: os.remove(p)
#             except: pass
#         print("removed:", p)
# PY
# pip cache purge || true

# echo "[C] toolchain"
# apt-get update -y
# apt-get install -y build-essential python3.11-dev

# echo "[D] build ulang paksa ABI FALSE"
# export CXXFLAGS="-D_GLIBCXX_USE_CXX11_ABI=0"
# pip install --no-cache-dir --no-build-isolation --no-binary=:all: causal-conv1d==1.5.0.post8
# pip install --no-cache-dir --no-build-isolation --no-binary=:all: mamba-ssm==2.3.1

# echo "[E] verifikasi import"
# python - <<'PY'
# import torch, causal_conv1d, mamba_ssm
# print("OK torch", torch.__version__, "abi", torch._C._GLIBCXX_USE_CXX11_ABI)
# print("OK causal_conv1d", getattr(causal_conv1d, "__version__", "n/a"))
# print("OK mamba_ssm", getattr(mamba_ssm, "__version__", "n/a"))
# PY

In [17]:
!apt-get update -y
!apt-get install -y build-essential python3.11-dev

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,528 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository '

In [19]:
# %%bash
# set -Eeuo pipefail

# on_err() {
#   ec=$?
#   echo ""
#   echo "[ERROR] line=$1 exit=$ec"
#   if [ -f /kaggle/working/train_kaggle.log ]; then
#     echo "----- tail /kaggle/working/train_kaggle.log -----"
#     tail -n 200 /kaggle/working/train_kaggle.log || true
#   fi
#   exit "$ec"
# }
# trap 'on_err $LINENO' ERR

# source /kaggle/working/.venv/bin/activate
# cd /kaggle/working/kcv-tts

# export HYDRA_FULL_ERROR=1
# export TORCH_DISTRIBUTED_DEBUG=DETAIL
# export PYTHONUNBUFFERED=1

# DATASET_NAME="${DATASET_NAME:-datasetku}"
# MODEL_NAME="${MODEL_NAME:-F5TTS_MAMBA_kaggle_full}"
# BATCH_SIZE_PER_GPU="${BATCH_SIZE_PER_GPU:-256}"
# MAX_SAMPLES="${MAX_SAMPLES:-1}"
# NUM_WORKERS="${NUM_WORKERS:-2}"
# EPOCHS="${EPOCHS:-1}"
# LEARNING_RATE="${LEARNING_RATE:-1e-5}"
# NUM_WARMUP_UPDATES="${NUM_WARMUP_UPDATES:-20}"
# GRAD_ACCUM="${GRAD_ACCUM:-4}"
# TARGET_SAMPLE_RATE="${TARGET_SAMPLE_RATE:-24000}"
# WANDB_PROJECT="${WANDB_PROJECT:-kcvanguard}"

# LOG_FILE=/kaggle/working/train_kaggle.log
# : > "$LOG_FILE"

# echo "[1/6] runtime check"
# which python
# python -V
# which accelerate
# accelerate launch --help >/dev/null

# echo "[2/6] dependency check (core only)"
# python - <<'PY'
# import importlib, torch
# print("torch", torch.__version__, "cuda", torch.version.cuda, "cxx11abi", torch._C._GLIBCXX_USE_CXX11_ABI)
# for m in ["accelerate", "hydra"]:
#     mod = importlib.import_module(m)
#     print("OK", m, getattr(mod, "__version__", "n/a"))
# print("deps_core_ok")
# PY

# echo "[3/6] mamba optional check"
# if python - <<'PY'
# import importlib, sys
# try:
#     importlib.import_module("causal_conv1d")
#     importlib.import_module("mamba_ssm")
#     print("mamba_stack_ok")
#     sys.exit(0)
# except Exception as e:
#     print("mamba_stack_off:", e)
#     sys.exit(1)
# PY
# then
#   USE_MAMBA=true
# else
#   USE_MAMBA=false
# fi
# echo "USE_MAMBA=$USE_MAMBA"

# echo "[4/6] dataset + pretrained"
# DATA_DIR="data/${DATASET_NAME}_pinyin"
# test -d "$DATA_DIR"

# CKPT_DIR="ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
# PRETRAIN_FILE="${CKPT_DIR}/pretrained_model_1250000.safetensors"

# if [ ! -f "$PRETRAIN_FILE" ]; then
#   ALT_PRETRAIN="$(ls -1 ckpts/*_vocos_pinyin_${DATASET_NAME}/pretrained_*.safetensors 2>/dev/null | head -n 1 || true)"
#   if [ -n "$ALT_PRETRAIN" ]; then
#     mkdir -p "$CKPT_DIR"
#     cp -f "$ALT_PRETRAIN" "$PRETRAIN_FILE"
#     echo "fallback pretrained copied: $ALT_PRETRAIN -> $PRETRAIN_FILE"
#   fi
# fi
# test -f "$PRETRAIN_FILE"

# echo "[5/6] logger + gpu"
# if [ -n "${WANDB_API_KEY:-}" ]; then
#   export WANDB_API_KEY
#   export WANDB_ENTITY="${WANDB_ENTITY:-}"
#   export WANDB_PROJECT
#   export WANDB_MODE=online
#   LOGGER_ARGS=(++ckpts.logger=wandb "++ckpts.wandb_project=${WANDB_PROJECT}" "++ckpts.wandb_run_name=${MODEL_NAME}-kaggle")
# else
#   export WANDB_MODE=offline
#   LOGGER_ARGS=(++ckpts.logger=tensorboard)
# fi

# GPU_COUNT="$(python - <<'PY'
# import torch
# print(torch.cuda.device_count())
# PY
# )"
# echo "GPU_COUNT=$GPU_COUNT"

# if [ "$GPU_COUNT" -ge 2 ]; then
#   export CUDA_VISIBLE_DEVICES=0,1
#   DIST_ARGS=(--multi_gpu --num_processes 2)
# elif [ "$GPU_COUNT" -eq 1 ]; then
#   export CUDA_VISIBLE_DEVICES=0
#   DIST_ARGS=(--num_processes 1)
# else
#   echo "No GPU detected"
#   exit 1
# fi

# if [ "$USE_MAMBA" = true ]; then
#   MODEL_ARGS=(++model.arch.use_mamba=true "++model.arch.mamba_layers=[2,3,6,7]")
# else
#   MODEL_ARGS=(++model.arch.use_mamba=false)
# fi

# echo "[6/6] training"
# set +e
# accelerate launch "${DIST_ARGS[@]}" --mixed_precision=fp16 \
#   src/f5_tts/train/train.py \
#   --config-name F5TTS_SANITY_HYBRID_1K.yaml \
#   "++datasets.name=${DATASET_NAME}" \
#   "++datasets.batch_size_per_gpu=${BATCH_SIZE_PER_GPU}" \
#   ++datasets.batch_size_type=frame \
#   "++datasets.max_samples=${MAX_SAMPLES}" \
#   "++datasets.num_workers=${NUM_WORKERS}" \
#   "++optim.epochs=${EPOCHS}" \
#   "++optim.learning_rate=${LEARNING_RATE}" \
#   "++optim.num_warmup_updates=${NUM_WARMUP_UPDATES}" \
#   "++optim.grad_accumulation_steps=${GRAD_ACCUM}" \
#   ++optim.max_grad_norm=1.0 \
#   ++optim.bnb_optimizer=False \
#   ++ckpts.log_samples=False \
#   ++ckpts.save_per_updates=200 \
#   ++ckpts.last_per_updates=100 \
#   ++ckpts.keep_last_n_checkpoints=3 \
#   "++model.name=${MODEL_NAME}" \
#   ++model.tokenizer=pinyin \
#   "++model.mel_spec.target_sample_rate=${TARGET_SAMPLE_RATE}" \
#   ++model.arch.dim=512 \
#   ++model.arch.depth=10 \
#   ++model.arch.heads=8 \
#   ++model.arch.text_dim=512 \
#   ++model.arch.checkpoint_activations=True \
#   "${MODEL_ARGS[@]}" \
#   "${LOGGER_ARGS[@]}" \
#   2>&1 | tee "$LOG_FILE"
# RC=${PIPESTATUS[0]}
# set -e

# if [ "$RC" -ne 0 ]; then
#   echo "Training gagal RC=$RC"
#   tail -n 200 "$LOG_FILE" || true
#   exit "$RC"
# fi

# echo "Training selesai RC=0"
# find "$CKPT_DIR" -maxdepth 3 -type f -name "model*.pt" -print | sort || true

In [ ]:
%%bash
set -Eeuo pipefail
source /kaggle/working/.venv/bin/activate
cd /kaggle/working/kcv-tts

export HYDRA_FULL_ERROR=1
export TORCH_DISTRIBUTED_DEBUG=DETAIL
export PYTHONUNBUFFERED=1
export CUDA_VISIBLE_DEVICES=0,1

# Tuning agresif (turunin kalau OOM)
DATASET_NAME="${DATASET_NAME:-datasetku}"
MODEL_NAME="${MODEL_NAME:-F5TTS_HEATUP_2GPU_$(date +%H%M%S)}"
WANDB_PROJECT="${WANDB_PROJECT:-kcvanguard}"

BATCH_SIZE_PER_GPU="${BATCH_SIZE_PER_GPU:-2048}"   # coba 2048 dulu
MAX_SAMPLES="${MAX_SAMPLES:-8}"                    # naikin packing
NUM_WORKERS="${NUM_WORKERS:-8}"
EPOCHS="${EPOCHS:-1}"
LEARNING_RATE="${LEARNING_RATE:-1e-5}"
NUM_WARMUP_UPDATES="${NUM_WARMUP_UPDATES:-100}"
GRAD_ACCUM="${GRAD_ACCUM:-1}"                      # biar update lebih berat per step

# Core deps check
python - <<'PY'
import torch, accelerate, hydra
print("torch", torch.__version__, "cuda", torch.version.cuda, "gpu_count", torch.cuda.device_count())
print("accelerate", accelerate.__version__)
print("hydra", hydra.__version__)
PY

# Mamba opsional; kalau error tetap non-mamba
if python - <<'PY'
import importlib, sys
try:
    importlib.import_module("causal_conv1d")
    importlib.import_module("mamba_ssm")
    sys.exit(0)
except Exception:
    sys.exit(1)
PY
then
  MODEL_ARGS=(++model.arch.use_mamba=true "++model.arch.mamba_layers=[2,3,6,7]")
  echo "USE_MAMBA=true"
else
  MODEL_ARGS=(++model.arch.use_mamba=false)
  echo "USE_MAMBA=false"
fi

# Fresh run, hindari mismatch resume lama
CKPT_DIR="ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
rm -rf "$CKPT_DIR"
mkdir -p "$CKPT_DIR"

if [ -n "${WANDB_API_KEY:-}" ]; then
  export WANDB_API_KEY
  export WANDB_ENTITY="${WANDB_ENTITY:-}"
  export WANDB_PROJECT
  export WANDB_MODE=online
  LOGGER_ARGS=(++ckpts.logger=wandb "++ckpts.wandb_project=${WANDB_PROJECT}" "++ckpts.wandb_run_name=${MODEL_NAME}")
else
  export WANDB_MODE=offline
  LOGGER_ARGS=(++ckpts.logger=tensorboard)
fi

accelerate launch --multi_gpu --num_processes 2 --mixed_precision=fp16 \
  src/f5_tts/train/train.py \
  --config-name F5TTS_SANITY_HYBRID_1K.yaml \
  "++datasets.name=${DATASET_NAME}" \
  "++datasets.batch_size_per_gpu=${BATCH_SIZE_PER_GPU}" \
  ++datasets.batch_size_type=frame \
  "++datasets.max_samples=${MAX_SAMPLES}" \
  "++datasets.num_workers=${NUM_WORKERS}" \
  "++optim.epochs=${EPOCHS}" \
  "++optim.learning_rate=${LEARNING_RATE}" \
  "++optim.num_warmup_updates=${NUM_WARMUP_UPDATES}" \
  "++optim.grad_accumulation_steps=${GRAD_ACCUM}" \
  ++optim.max_grad_norm=1.0 \
  ++optim.bnb_optimizer=False \
  ++ckpts.log_samples=False \
  ++ckpts.save_per_updates=100 \
  ++ckpts.last_per_updates=50 \
  ++ckpts.keep_last_n_checkpoints=3 \
  "++model.name=${MODEL_NAME}" \
  ++model.tokenizer=pinyin \
  "${MODEL_ARGS[@]}" \
  "${LOGGER_ARGS[@]}"

echo "Done: $CKPT_DIR"

## 6) Lokasi Checkpoint dan Quick Inference (Opsional)

In [ ]:
from pathlib import Path
ckpt_dir = Path(f'/kaggle/working/kcv-tts/ckpts/{MODEL_NAME}_vocos_pinyin_{DATASET_NAME}')
print('Checkpoint dir:', ckpt_dir)
for p in sorted(ckpt_dir.glob('model*.pt'))[-5:]:
    print(p)

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/.venv/bin/activate
cd /kaggle/working/kcv-tts

# Contoh quick inference setelah model_last.pt tersedia.
# Ganti ref_audio/ref_text sesuai data kamu.
MODEL_CFG="$(ls -1t ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}/*/.hydra/config.yaml | head -n 1)"
python src/f5_tts/infer/infer_cli.py \
  -m F5TTS_Small \
  -mc "$MODEL_CFG" \
  -p ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}/model_last.pt \
  -v data/${DATASET_NAME}_pinyin/vocab.txt \
  -r /kaggle/input/your-dataset/example.wav \
  -s "Ini teks referensi." \
  -t "Halo semua kenalin Aku Sedang Makan" \
  -o /kaggle/working \
  -w infer_kaggle.wav \
  --device cuda --nfe_step 24 --cfg_strength 2.0